In [84]:
from openai import OpenAI
import json
import requests

In [85]:
id = "lmiv1c6fvhq727"

In [86]:
runpod_url = f"https://{id}-8000.proxy.runpod.net"

In [87]:
health_url = f"{runpod_url}/health"

In [88]:
health_check_res = requests.get(health_url)
if health_check_res.status_code != 200:
    print("헬스 체크 실패")
    exit()

print(health_check_res.json())

{'status': 'ok'}


In [89]:
model_list_url = f"{runpod_url}/v1/models"

In [90]:
model_list_res = requests.get(model_list_url)

if model_list_res.status_code != 200:
    print("모델 목록 조회 실패")
    exit()

model_list = model_list_res.json()

print(model_list)

model_name = model_list['data'][0]['id']

{'object': 'list', 'data': [{'id': 'google/gemma-4-31B-it', 'object': 'model', 'created': 1776225444, 'owned_by': 'vllm', 'root': 'google/gemma-4-31B-it', 'parent': None, 'max_model_len': 65536, 'permission': [{'id': 'modelperm-aaadb4b27d6410bf', 'object': 'model_permission', 'created': 1776225444, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [91]:
chat_url = f"{runpod_url}/v1/chat/completions"

In [92]:
import base64

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [93]:
def build_messages(image_path: str, text: str) -> list:
    ext = image_path.rsplit(".", 1)[-1].lower()
    mime_map = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp", "gif": "image/gif"}
    mime_type = mime_map.get(ext, "image/jpeg")

    b64 = encode_image(image_path)

    print(f"{b64[:100]}...")

    return [{
        "role": "user",
        "content": [
            {"type": "text", "text": text},
            {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{b64}"}}
        ]
    }]



In [100]:
image_path = "D:\TEST\orchestrator_loop\image_data\\cat.jpg"

In [103]:
messages = build_messages(image_path, "이미지를 자세히 설명해줘명해줘")

iVBORw0KGgoAAAANSUhEUgAAAeAAAAFACAIAAADrqjgsAAAgAElEQVR4nKy8aW9lyXYltocYzrkDySRzqHp60ntPslpSu+F2w4YB...


In [104]:
payload = {
    "model" : model_name,
    "messages": messages,
    "max_tokens": 4000,
    "stream": False
}

In [97]:
res = requests.post(chat_url, json=payload)

In [98]:
print(res.json())


{'id': 'chatcmpl-bde02c2fa339f91b', 'object': 'chat.completion', 'created': 1776225465, 'model': 'google/gemma-4-31B-it', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '제공해주신 영수증의 내용은 다음과 같습니다.\n\n**[기본 정보]**\n*   **상호명:** 세븐일레븐 문정수정점 (#18308)\n*   **일시:** 2020년 6월 9일 (화) 20:59:47\n*   **결제 수식:** 현금 (자진발급)\n\n**[구매 상품]**\n1.  **라라스윗 바닐라 파인트 474ml:** 6,900원 (1개)\n2.  **라라스윗 초코 파인트 474ml:** 6,900원 (1개)\n3.  **비닐봉투 보증금:** 20원 (1개)\n\n**[금액 상세]**\n*   **공급가액:** 12,545원\n*   **부가세:** 1,255원\n*   **봉투보증금:** 20원\n*   **합계 금액:** **13,820원**', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'stop', 'stop_reason': 106, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 287, 'total_tokens': 538, 'completion_tokens': 251, 'prompt_tokens_details': None}, 'prompt_logprobs': None, 'prompt_token_ids': None, 'kv_transfer_params': None

In [105]:
payload['stream'] = True

res = requests.post(chat_url, json=payload)

is_reasoning = False

for line in res.iter_lines():
    if line:
        decoded = line.decode('utf-8')

        if decoded.startswith('data: '):
            data = decoded.split('data: ')[1]

            if data == '[DONE]':
                break;
            
            json_data = json.loads(data)
            
            tokens = json_data['choices'][0]['delta']

            if tokens.get('reasoning', None):
                if not is_reasoning: 
                    print("<REASONING>")
                    is_reasoning = True
                print(tokens['reasoning'], end='', flush=True)
            
            if tokens.get('content', None):
                if is_reasoning:
                    print("\n<RESPONSE>")
                    is_reasoning = False
                print(tokens['content'], end='', flush=True)

이 이미지는 유머러스하게 합성된 사진으로, 체육관(헬스장)에서 벤치 프레스 운동을 하고 있는 뚱뚱한 오렌지색 고양이를 보여줍니다. 자세한 설명은 다음과 같습니다.

**1. 중심 대상 (고양이):**
*   **외형:** 매우 통통한 체형의 오렌지색 줄무늬 고양이가 벤치에 누워 있습니다. 특히 볼록하게 튀어나온 배가 강조되어 있어 코믹한 느낌을 줍니다.
*   **표정:** 눈을 지그시 감고 있으며, 매우 만족스럽거나 혹은 운동의 고단함에 체념한 듯한 평온한 표정을 짓고 있습니다.
*   **동작:** 두 앞발로 바벨을 잡고 들어 올리려는 자세를 취하고 있습니다.

**2. 운동 기구 및 설정:**
*   **바벨:** 고양이의 가슴 위에 무거운 원판이 끼워진 바벨이 놓여 있습니다.
*   **벤치:** 고양이는 검은색 가죽 소재의 웨이트 트레이닝 벤치 위에 누워 있습니다.
*   **배경:** 배경에는 실제 헬스장 모습이 보입니다. 뒤편에 다른 사람들이 운동하고 있는 다리와 운동 기구들이 흐릿하게(아웃포커싱) 처리되어 있어, 고양이에게 시선이 집중되도록 합니다.

**3. 전체적인 분위기:**
*   이 사진은 실제 상황이 아니라 디지털 합성을 통해 만들어진 **'밈(Meme)' 스타일의 이미지**입니다.
*   고양이의 포동포동한 모습과 격렬한 웨이트 트레이닝이라는 상반된 요소가 결합되어 보는 사람에게 웃음을 주는 귀엽고 우스꽝스러운 분위기를 자아냅니다.